# unit04 レッスン: テーブルの特徴量エンジニアリング

**ここまでの2日で確定したこと** — Day2(unit02)で **信じられる CV** を作った(`GroupKFold(5)` を
`groups=train["product_key"]` で切る / `list_price` と `discount_rate` はリーク列なので使わない /
目的変数は `np.log1p(price)`、評価は log 空間の RMSE = RMSLE)。
Day3(unit03)で **その上にモデルを載せた**(LightGBM + early stopping、重要度でリークを点検)。

**今日は特徴量でスコアを押し上げる。** モデルは昨日のまま、動かすのは**入力の側**だ。

**データは昨日・一昨日と同じ**(`unit02-validation-and-leakage/data/`)。同じコンペを3日かけて改善していく:

| Day | やること | 状態 |
|---|---|---|
| Day2 (unit02) | 信じられる CV を作る | **済** |
| Day3 (unit03) | その上にモデルを載せる | **済**(LightGBM で CV RMSE 0.7074) |
| **Day4 (このユニット)** | **特徴量でスコアを押し上げる** | ← いまここ |

## このレッスンを終えると作れるようになるもの

1. カテゴリ変数の4つの表現(**one-hot / ordinal / count / LightGBM の native categorical**)を、
   **それぞれが何を仮定しているか**で選び分けられる。高カーディナリティ列で one-hot が壊れることを shape で説明できる
2. **target encoding** を作れる。そして**素朴に作ると必ずリークする**ことを実測で示し、
   **OOF target encoding** に直せる。スムージングの式と意味を説明できる
3. `groupby().agg()` / `transform()` / `merge()` を使い分けられる(**行数が変わるか変わらないか**で選ぶ)。
   `dt` アクセサで日付から特徴量を作れる
4. `Pipeline` と `ColumnTransformer` で前処理をモデルの一部にし、**前処理リークを構造的に潰せる**
5. そして今日いちばん大事なこと — **CV が跳ね上がったときに、喜ぶ前に疑える**

所要の目安: **90〜120分**。このあと演習 `ex01`〜`ex04` が続く。

## このレッスンの読み方

セルは**上から順に**実行する。構成は概念ごとに次の8ステップの繰り返し:

| 記号 | 内容 |
|---|---|
| ① | なぜこれを学ぶのか(実務のどこで使うか) |
| ② | 解説(C# との対応表・API 表) |
| ③ | **見る** — 完成コードを実行して結果を見る |
| ④ | **予測する** — 次のセルの結果を、実行する前に予想する |
| ⑤ | **変えてみる** — ③ の条件を変えて実行し、予測と照合する |
| ⑥ | **書いてみる**(指示) |
| ⑦ | **書いてみる**(君が書くセル) |
| ⑧ | チェックポイント(即時採点) |

⑦ を飛ばしても後続セルは動く(⑧ が `[NG]` を出すだけ)。詰まったら ⑤ に戻ればよい。


In [ ]:
# ===== セットアップ: このセルを最初に1回だけ実行する =====
import os

# 小さいデータではスレッドを増やすほど遅くなる。1に固定すると速く、結果も完全に再現する。
os.environ.setdefault("OMP_NUM_THREADS", "1")

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

# データは unit02 / unit03 と同じもの。
# notebook をこのユニット直下で開いても、リポジトリのルートで開いても動くようにする。
DATA = Path("../unit02-validation-and-leakage/data")
if not (DATA / "train.csv").exists():
    DATA = Path("courses/kaggle-sprint/unit02-validation-and-leakage/data")
assert (DATA / "train.csv").exists(), f"train.csv が見つかりません: {DATA.resolve()}"

print("pandas:", pd.__version__, "/ numpy:", np.__version__,
      "/ scikit-learn:", sklearn.__version__, "/ lightgbm:", lgb.__version__)
print("DATA =", DATA.resolve())

train = pd.read_csv(DATA / "train.csv", parse_dates=["collected_at"])

# ---------- Day2 / Day3 で確定した設定をそのまま持ってくる ----------
ORIGIN = train["collected_at"].min()                       # 起点(test でもこの起点を使う)
train["days"] = (train["collected_at"] - ORIGIN).dt.days   # 起点からの経過日数
FEAT_NUM = ["brand_tier", "views", "title_len", "days"]    # 数値列
FEAT_CAT = ["category", "site", "condition"]               # 低カーディナリティのカテゴリ列
y = np.log1p(train["price"].to_numpy(dtype=float))         # 目的変数は log 空間(= RMSLE)
groups = train["product_key"]                              # 同一商品は同じ fold に閉じ込める

gkf = GroupKFold(n_splits=5)
FOLDS = list(gkf.split(train, y, groups))                  # 分割は1回だけ作って全実験で使い回す

LGB_BASE = dict(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=20,
                subsample=0.9, subsample_freq=1, colsample_bytree=0.9, reg_lambda=1.0,
                random_state=42, n_jobs=1, verbose=-1)


def lgbm(**kw):
    """LGB_BASE の設定に上書きを重ねた LGBMRegressor を作る(未学習)。"""
    return LGBMRegressor(**{**LGB_BASE, **kw})


def cv_rmse(model, X, y=y, folds=FOLDS, return_oof=False):
    """GroupKFold の OOF 予測を作り、log 空間の RMSE を返す。
    model は fit / predict を持つものなら何でもよい(Pipeline でも可)。"""
    oof = np.zeros(len(y))
    for idx_tr, idx_va in folds:
        m = clone(model)
        m.fit(X.iloc[idx_tr], y[idx_tr])
        oof[idx_va] = m.predict(X.iloc[idx_va])
    score = float(np.sqrt(np.mean((oof - y) ** 2)))
    return (score, oof) if return_oof else score


print("\ntrain:", train.shape, " y:", y.shape, " fold 数:", len(FOLDS))
print("train 期間:", train["collected_at"].min().date(), "〜", train["collected_at"].max().date())
print("水準数: ", {c: int(train[c].nunique()) for c in FEAT_CAT + ["model_code", "product_key"]})


# ---------- 採点ヘルパー(中身は読まなくてよい) ----------
def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok


def check_frame(name, actual, shape=None, columns=None, hint=""):
    """DataFrame の形と列名を採点する。DataFrame でなくても例外にしない。"""
    if not isinstance(actual, pd.DataFrame):
        print(f"[NG] {name}: 期待値 pandas.DataFrame(shape={shape}, columns={columns}) / 実際 {type(actual).__name__}")
        if hint:
            print(f"     ヒント: {hint}")
        return False
    problems = []
    if shape is not None and tuple(actual.shape) != tuple(shape):
        problems.append(f"shape の期待値 {tuple(shape)} / 実際 {tuple(actual.shape)}")
    if columns is not None and list(actual.columns) != list(columns):
        problems.append(f"列名の期待値 {list(columns)} / 実際 {list(actual.columns)}")
    if problems:
        print(f"[NG] {name}: " + " | ".join(problems))
        if hint:
            print(f"     ヒント: {hint}")
        return False
    print(f"[OK] {name}: 正解!")
    return True


def call_safely(fn, *args, **kwargs):
    """未完成の関数を呼んでも notebook が止まらないようにするラッパ。例外なら None を返す。"""
    if not callable(fn):
        return None
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print(f"     (関数の中で例外が出ました → {type(e).__name__}: {e})")
        return None


def param_of(model, key):
    """model.get_params()[key] を安全に取り出す。取れなければ None。"""
    try:
        return model.get_params().get(key)
    except Exception:
        return None


def series_at(s, key):
    """Series[key] を安全に float で取り出す。取れなければ None。"""
    try:
        return float(s[key])
    except Exception:
        return None


def frame_stat(df, col, how):
    """df[col] の統計を安全に取り出す。取れなければ None。"""
    if not isinstance(df, pd.DataFrame) or col not in df.columns:
        return None
    try:
        return {"sum": float(df[col].sum()), "mean": float(df[col].mean()),
                "max": float(df[col].max()), "min": float(df[col].min())}[how]
    except Exception:
        return None


print("\nセットアップ完了。ヘルパー: lgbm / cv_rmse / check / check_frame / call_safely / param_of / series_at / frame_stat")


---
# 概念1 — カテゴリ変数をどう数値にするか

## ① なぜ: モデルは行列しか食べられない。文字列は必ず数値に変換される

`category` は `"家電"` `"ファッション"` … という**文字列**で、`site` も `condition` も同じだ。
だがどんなモデルも最終的には**数値の行列**しか受け取れない。だから必ずどこかで変換が起きる。

問題は、この変換が**中立ではない**ことだ。同じ列でも変換方法を変えるだけでスコアが動く。
今日の実測で言うと、`category` / `site` / `condition` の3列を **one-hot にするか整数にするか**だけで、
線形モデルの CV は **0.6379 と 1.0750**(約1.7倍)まで開く。モデルを変えるより大きい差だ。

実務でも、この判断は毎回やってくる。「ユーザーの居住都道府県(47水準)」「商品カテゴリ(3階層で1000水準)」
「ブラウザの User-Agent(数万水準)」。**水準数**と**使うモデル**の組み合わせで、正解の変換は変わる。
まずは4つの選択肢と、それぞれが**何を暗黙に仮定しているか**を押さえる。


## ② 解説: 4つの表現と、その暗黙の仮定

### 比較表 — 「何を仮定しているか」で選ぶ

| 手法 | 何を作るか | **暗黙の仮定** | 列の増え方 | 木モデル | 線形モデル |
|---|---|---|---|---|---|
| **one-hot** | 水準ごとに 0/1 の列を1本ずつ | 水準どうしに**順序も距離も無い**(全部が対等) | **+ 水準数**。爆発する | **不利になりやすい**(下記) | **必須**。これ以外に選択肢がない |
| **ordinal (label)** | 水準に `0,1,2,...` の整数を振る | **整数の大小に意味がある**(実際には無いのに順序を捏造する) | ± 0 | **使える**。木は「≤ k」で切るので、任意の集合分割を階段状に再現できる | **破綻する**。「家電 = 0、ホビー = 2 だから2倍」という係数を学んでしまう |
| **count / frequency** | 水準を**出現回数**に置き換える | **珍しさ**が目的変数と関係する | ± 0 | 使える | 使える |
| **native categorical**<br>(LightGBM) | 変換しない。`category` dtype のまま渡す | 特になし(木が集合の分割を直接学ぶ) | ± 0 | **有利**(高カーディナリティで特に) | 使えない |
| **target encoding** | 水準を**その水準の目的変数の平均**に置き換える | 水準ごとの平均が**安定して推定できる**ほど件数がある | ± 0 | **最強** | 強い |

target encoding だけは別格に強く、別格に危険なので **概念2 まるごと**を使って扱う。

### なぜ木モデルに one-hot が不利なのか

10水準のカテゴリを one-hot にすると、木が使えるのは `「家電か否か」` のような **1対9の分割だけ**になる。
`{家電, ホビー} vs 残り` という分割をしたければ、木を**2段**掘る必要がある。
つまり **1列あたりの情報が薄まり、同じ表現に到達するのにより深い木が必要になる**。
おまけに `subsample` / `colsample_bytree`(列のランダム抽出)を掛けると、
薄い列がバラバラに選ばれて余計に効率が落ちる。

一方 native categorical(LightGBM)は「カテゴリの集合を2つに分ける」分割を**1回で**学ぶので、この損がない。

```
one-hot + 木                       native categorical + 木
  [is_家電 == 1?]                    [category ∈ {家電, ホビー}?]
   /        \                          /            \
 ...   [is_ホビー == 1?]              葉             葉        ← 1回の分割で済む
          /      \
        葉       葉                 ← 同じ分割に2段かかる
```

### 逆に、線形モデルには one-hot が必須

線形モデルは `y = w1·x1 + w2·x2 + ...` という**重み付き和**しか作れない。
`category` に `0,1,2,3,4` という整数を振ると、モデルは「`category` が1増えると価格が `w` 増える」としか表現できない。
`家電=0, ファッション=1, ホビー=2` の並びに意味は無いのだから、これは**ほぼ必ず間違い**になる。
one-hot にして初めて「水準ごとに独立した切片」を持てる。

### C# との対応

| C# での構図 | pandas / sklearn |
|---|---|
| `enum Category { 家電, ファッション, ... }` を `bool Is家電; bool Isファッション; ...` に展開する | one-hot |
| `(int)category` をそのまま数値として計算に使う(**意味的に誤り**) | ordinal encoding |
| `Dictionary<string,int> freq` を作って `freq[cat]` を引く | count encoding |
| `enum` のまま渡し、受け手が集合として扱う | native categorical |

### API 一覧

| 用途 | API | 戻り値 | 注意 |
|---|---|---|---|
| one-hot(pandas) | `pd.get_dummies(df, columns=[...])` | DataFrame(列が増える) | **train と test で列がズレる**。test 側は `reindex(columns=X.columns, fill_value=False)` で揃える |
| one-hot(sklearn) | `OneHotEncoder(handle_unknown="ignore")` | 疎行列 or ndarray | `handle_unknown="ignore"` で**未知の水準は全部 0 の行**になる。Pipeline に入れられるのが利点 |
| ordinal | `OrdinalEncoder()` の `fit_transform(df[cols])` | ndarray(列数そのまま) | 既定では未知の水準で例外。`handle_unknown="use_encoded_value", unknown_value=-1` を付ける |
| count | `s.value_counts()` → `s.map(...)` | Series | `value_counts()` は **index = 水準, 値 = 件数** の Series。`map` はその Series を辞書として引く(C# の `dict[key]` を列全体に適用) |
| native categorical | `s.astype("category")` | category dtype の Series | LightGBM が自動でカテゴリ列として扱う。**test 側は水準の一覧も揃える**(`pd.Categorical(s, categories=...)`) |
| 数値列かの判定 | `pd.api.types.is_numeric_dtype(s)` | bool | **pandas 3.0 では文字列列の dtype は `object` ではなく `str`。`dtype == object` 判定は必ず False になる**(unit03 で扱った罠) |


In [ ]:
# GOAL: 同じ3列を3通りに変換し、同じ CV で測って「変換だけでスコアが動く」ことを見る

from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# STEP 1: one-hot — カテゴリ列を 0/1 の列に開く。列がどれだけ増えたかを必ず shape で確認する
X_oh = pd.get_dummies(train[FEAT_NUM + FEAT_CAT], columns=FEAT_CAT)
print("元:", train[FEAT_NUM + FEAT_CAT].shape, "→ one-hot:", X_oh.shape,
      f"(+{X_oh.shape[1] - len(FEAT_NUM) - len(FEAT_CAT)} 列)")
print("  水準数の合計:", sum(int(train[c].nunique()) for c in FEAT_CAT),
      "= 増えた列 + 元のカテゴリ列3本")
print("  列:", list(X_oh.columns))

# STEP 2: ordinal — 水準に 0,1,2,... の整数を振る。列数は変わらない
X_ord = train[FEAT_NUM + FEAT_CAT].copy()
enc = OrdinalEncoder()
X_ord[FEAT_CAT] = enc.fit_transform(X_ord[FEAT_CAT])   # fit_transform は「覚えて変換する」
print("\nordinal:", X_ord.shape, "  category に振られた整数:",
      {v: i for i, v in enumerate(enc.categories_[0])})
print("  ↑ この 0,1,2,3,4 の並びに意味は無い(アルファベット/文字コード順に振られただけ)")

# STEP 3: native categorical — 変換しない。dtype を category にするだけ
X_cat = train[FEAT_NUM + FEAT_CAT].copy()
for c in FEAT_CAT:
    X_cat[c] = X_cat[c].astype("category")
print("\nnative categorical:", X_cat.shape, " dtype:",
      {c: str(X_cat[c].dtype) for c in FEAT_CAT})

# STEP 4: 同じ CV・同じモデルで測る。動かしたのは「表現」だけ
print(f"\n{'表現':<24}{'列数':>6}{'LightGBM':>12}{'Ridge(標準化)':>16}")
print("-" * 60)
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
for name, Xd, use_ridge in [("one-hot", X_oh, True), ("ordinal", X_ord, True),
                            ("native categorical", X_cat, False)]:
    r = f"{cv_rmse(ridge, Xd):>16.6f}" if use_ridge else f"{'(使えない)':>14}"
    print(f"{name:<20}{Xd.shape[1]:>6}{cv_rmse(lgbm(), Xd):>12.6f}{r}")

print("\n→ LightGBM は3つとも 0.707〜0.711 でほぼ横並び(木は表現に鈍い)")
print("→ Ridge は one-hot 0.637878 に対し ordinal 1.075022。定数予測(1.111857)とほぼ同じところまで壊れた")


## ④ 予測: 高カーディナリティ列を足すと、one-hot はどうなる?

いま使っているカテゴリ列は `category`(5水準)/ `site`(4水準)/ `condition`(4水準)と小さい。
ここに **`model_code`(型番、756水準)** を足す。

> `model_code` は unit02 で「商品をほぼ一意に指すので使うと丸暗記になる」と確認した列だ。
> 今日は**特徴量の表現の実験材料**として意図的に使う。CV は `GroupKFold(product_key)` のままなので、
> 検証 fold の型番は学習側に出てこない(= 丸暗記は効かない)。

実行する前に予測しよう。

1. `pd.get_dummies` に `model_code` を含めると、**列数**はいくつになる? (いまは 17 列)
2. その one-hot 版の **LightGBM の CV スコア**は、`model_code` 無し(0.711313)からどう動く?
   良くなる? 悪くなる? **まったく動かない**?
3. `astype("category")` で native categorical にした版(列数は 8)はどうなる?
4. `Ridge` に 700 列超の one-hot を渡したらどうなる? (いまは 0.637878)

> ヒント: 756 水準の型番は、1つあたり train に平均 **2.2 行**しか無い(1661 行 ÷ 756 水準)。
> 「1本の列に 1661 行中 2 行だけ 1 が立っている」列が 756 本並ぶ、という絵を思い浮かべよう。
> 木は `min_child_samples=20`(葉に最低20行)で動いている。


In [ ]:
# GOAL: 756 水準の列に one-hot を掛けると何が起きるかを、列数・スコア・時間の3点で見る

import time

HIGH = ["model_code"]
print("model_code の水準数:", int(train["model_code"].nunique()),
      " 1水準あたりの平均行数:", round(len(train) / train["model_code"].nunique(), 2))
print("出現回数の分布(何行ある型番が何個あるか):",
      train["model_code"].value_counts().value_counts().sort_index().to_dict())

# STEP 1: one-hot に model_code を含める
X_oh_mc = pd.get_dummies(train[FEAT_NUM + FEAT_CAT + HIGH], columns=FEAT_CAT + HIGH)
print("\none-hot:", X_oh.shape, "→ model_code を足すと:", X_oh_mc.shape,
      f"({X_oh_mc.shape[1] / X_oh.shape[1]:.0f}倍)")

# STEP 2: native categorical に model_code を含める(列は1本増えるだけ)
X_cat_mc = X_cat.copy()
X_cat_mc["model_code"] = train["model_code"].astype("category")
print("native categorical:", X_cat.shape, "→", X_cat_mc.shape)

# STEP 3: 同じ CV で測る。時間も測る
print(f"\n{'表現':<30}{'列数':>6}{'CV RMSE':>11}{'秒':>7}")
print("-" * 56)
for name, Xd, model in [("one-hot(model_code なし)", X_oh, "lgbm"),
                        ("one-hot(model_code あり)", X_oh_mc, "lgbm"),
                        ("native cat(model_code なし)", X_cat, "lgbm"),
                        ("native cat(model_code あり)", X_cat_mc, "lgbm"),
                        ("Ridge + one-hot(なし)", X_oh, "ridge"),
                        ("Ridge + one-hot(あり)", X_oh_mc, "ridge")]:
    t0 = time.time()
    s = cv_rmse(lgbm() if model == "lgbm" else ridge, Xd)
    print(f"{name:<26}{Xd.shape[1]:>6}{s:>11.6f}{time.time() - t0:>7.2f}")

print("\n→ one-hot は列を 17 → 773 に爆発させたのに、LightGBM のスコアは小数第6位まで完全に同一。")
print("  1本あたり 2 行しか立っていない列では、min_child_samples=20 の葉が作れず一度も分割に使われない。")
print("  = 情報を1ミリも足さずにメモリと時間だけ食った")
print("→ Ridge は 0.637878 → 0.745690 と悪化。756 本の『ほぼ全部ゼロの列』に係数を付けようとして過学習した")
print("→ native categorical は列を1本足しただけ。木が集合の分割を直接学ぶので破綻しない")


## ⑥ 書いてみる: count(frequency)エンコーディングを作る

②の表に出てきた **count encoding** をまだ実際に作っていない。ここで書く。

考え方は単純で、**水準を「その水準が何回出てくるか」に置き換える**。
「よく出てくる型番 = 定番商品」「1回しか出てこない型番 = レア」という**珍しさ**が
目的変数と関係するなら効く、という仮定に立っている。列は1本も増えない。

次のセルで4つ作ろう。

| 変数 | 中身 |
|---|---|
| `X_count` | `train[FEAT_NUM]` のコピーに、`category` / `site` / `condition` / `model_code` の**出現回数**の列を足したもの。列名は元の列名 + `"_count"`。**この順**で足すこと(結果は 8 列) |
| `score_count` | `X_count` の CV RMSE(`cv_rmse(lgbm(), X_count)`) |
| `mc_count_max` | `model_code_count` 列の**最大値**(`int` にすること) |
| `n_singleton` | `model_code_count` が **1 の行**が何行あるか(`int`) |

使う道具:

- `train[c].value_counts()` — **index = 水準、値 = 件数**の Series を返す
- `train[c].map(その Series)` — 各行の値をキーにして Series を引く(C# の `freq[key]` を列全体に適用するのと同じ)
- `for c in FEAT_CAT + ["model_code"]:` — リストの `+` は連結
- `int(...)` / `(条件).sum()` — ブール Series の `.sum()` は True の個数

4〜6行で書ける。`n_singleton` は「train に1行しか無い型番の行数」であり、
**概念2 でそのまま重要な意味を持つ数**になる。


In [ ]:
X_count = None       # ここに書く(ヒント: train[FEAT_NUM].copy() から始めて、for ループで列を足す)
score_count = None   # ここに書く(ヒント: cv_rmse(lgbm(), X_count))
mc_count_max = None  # ここに書く(ヒント: X_count["model_code_count"].max() を int に)
n_singleton = None   # ここに書く(ヒント: (X_count["model_code_count"] == 1).sum() を int に)

print("X_count     :", None if X_count is None else X_count.shape)
if isinstance(X_count, pd.DataFrame):
    print("列          :", list(X_count.columns))
    print(X_count.head(3).to_string())
print("score_count :", score_count)
print("mc_count_max:", mc_count_max, " n_singleton:", n_singleton)


In [ ]:
# ===== チェックポイント A: カテゴリ変数の表現 =====
check_frame("A-1 X_count の形と列名", X_count, shape=(1661, 8),
            columns=["brand_tier", "views", "title_len", "days",
                     "category_count", "site_count", "condition_count", "model_code_count"],
            hint="train[FEAT_NUM].copy() に4本の *_count 列を足して 8 列。"
                 "列が 11 列なら元のカテゴリ列を消し忘れている。列名は元の列名 + '_count'。")

check("A-2 count エンコードでの CV RMSE", score_count, 0.711380,
      hint="cv_rmse(lgbm(), X_count) の戻り値。0.707434 なら X_cat を渡している。"
           "None のままなら X_count が作れていない。")

check("A-3 model_code_count の最大値", mc_count_max, 4,
      hint="同じ型番が train に最大何行あるか。train['model_code'].value_counts().max() と同じ値。")

check("A-4 出現1回だけの型番の行数", n_singleton, 204,
      hint="(X_count['model_code_count'] == 1).sum()。1661 行中この数だけが『train にたった1行しかない型番』。")

print("\n(4つとも [OK] になったら次の概念へ)")
print("※ count エンコードの CV は 0.711380 で、native categorical の 0.707434 に負けている。")
print("  『珍しさが価格と関係する』という仮定が、このデータでは成り立っていなかった、というだけのこと。")
print("  効かない手法を1つ潰したのも立派な進捗で、これを CV で判定できるのが Day2 の成果だ。")


---
# 概念2 — target encoding とリークの回避(今日の山場)

## ① なぜ: 「最も効く特徴量」であり「最も事故る特徴量」

target encoding は、カテゴリを**そのカテゴリの目的変数の平均**で置き換える手法だ。
`category` を `"家電"` ではなく `9.7`(家電の平均 log 価格)にする。

これは Kaggle のテーブルコンペで**最も広く使われ、最も上位に効く**特徴量の作り方であり、
実務でも「店舗ID → その店舗の平均売上」「ユーザーID → そのユーザーの平均購入額」として日常的に使われる。
高カーディナリティの列を、列を1本も増やさずに、目的変数と直接結びついた強い数値に変換できるからだ。

そして同時に、**Kaggle で最も多くの人が事故る特徴量**でもある。
理由は一言で言える — **素朴に作ると、その行自身の答えを見た値になる**。

今日は順番に、**(1) 素朴に作って CV が不自然に良くなることを実測し、(2) OOF に直して現実に戻す**。
unit02 で入れた規律 —— **「良すぎるスコアは喜ぶ前に疑う」** —— が、ここで実戦になる。


## ② 解説: 平均を「どのデータから計算したか」が全て

### 素朴な target encoding(= リークする)

```
1. train 全体で、model_code ごとの y の平均を計算する      ← ここに検証 fold の答えが入っている
2. その平均を train の各行に貼る
3. その列を特徴量にして CV を回す
```

問題は 1 と 3 の関係だ。**CV の検証 fold の行も、平均の計算に参加している。**
極端な例として、train に**1行しかない型番**を考えよう(概念1 で数えた通り 204 行ある)。
その行の target encoding の値は

```
(その行の y) / 1 = その行の y そのもの
```

つまり **目的変数をそのまま特徴量の列に書き写している**。これでモデルが当たらないわけがない。
これは unit02 の `discount_rate` と同じ「目的変数から作られた列」であり、種類としては同じ事故だ。

### OOF target encoding(= 正しい)

直し方も一言だ。**平均を作るときに、貼る相手を混ぜない。**

```
fold k について:
    学習側 (idx_tr) のデータ「だけ」で model_code ごとの平均を計算する
    その平均を、検証側 (idx_va) の行に貼る
    学習側に無かった水準は、学習側の全体平均(prior)で埋める
```

CV ループと**同じ分割**で統計を作るので、検証 fold の答えは一切使われない。
unit02 で書いた OOF 予測のループと**構造が完全に同じ**であることに注目してほしい
(「fold ごとに学習側だけから何かを作り、検証側に書き込む」)。

### スムージング — 件数の少ない水準を全体平均に寄せる

素朴でも OOF でも、**件数が少ない水準の平均は信用できない**。1件の平均は平均ではない。
そこで「全体平均 `prior` を、あたかも `m` 件ぶん観測したかのように混ぜる」:

$$\text{smoothed}_k = \frac{\sum_{i \in k} y_i + \text{prior} \times m}{n_k + m}$$

| 状況 | 分子・分母 | 結果 |
|---|---|---|
| `n_k` が `m` より**ずっと大きい** | `prior × m` の影響が相対的に小さい | **その水準の平均**にほぼ一致 |
| `n_k` が `m` より**ずっと小さい** | `prior × m` が支配する | **全体平均**にほぼ一致 |
| `m = 0` | そのまま | スムージング無し(素朴な平均) |

`m` は「この水準を信じ始めるのに何件必要か」を表すツマミだ。ベイズの事前分布そのものだが、
実装としてはただの重み付き平均でよい。

> **注意: スムージングはリークを直さない。** 平均を**どのデータから計算したか**を変えていないからだ。
> スムージングは「少数水準のノイズ」への対策であって、「自分の答えを見ている」問題への対策ではない。
> 今日この2つを混同しないことが、いちばん大事な区別になる。

### API 一覧

| 用途 | API | 戻り値 | 注意 |
|---|---|---|---|
| 水準ごとの合計 | `pd.Series(y).groupby(keys).sum()` | index = 水準の Series | `keys` は ndarray でも Series でもよい。**長さが `y` と同じ**であること |
| 水準ごとの件数 | `pd.Series(y).groupby(keys).size()` | 同上 | `count()` は欠損を除く、`size()` は除かない |
| 対応表を貼る | `s.map(mapping)` | Series | `mapping` に無いキーは `NaN` → **必ず `fillna(prior)`** |
| sklearn 版 | `sklearn.preprocessing.TargetEncoder` | 変換器(estimator) | `fit_transform` の中で**内部的に交差適合**する。ただし後述の落とし穴あり(概念4) |

### C# で書くなら

```csharp
// 素朴版(リークする)
var map = rows.GroupBy(r => r.ModelCode)
              .ToDictionary(g => g.Key, g => g.Average(r => r.Y));
foreach (var r in rows) r.Te = map[r.ModelCode];   // ← 自分自身も map の材料に入っている

// OOF 版
foreach (var (trainIdx, validIdx) in folds) {
    var map = trainIdx.Select(i => rows[i])        // ← 学習側だけから作る
                      .GroupBy(r => r.ModelCode)
                      .ToDictionary(g => g.Key, g => g.Average(r => r.Y));
    var prior = trainIdx.Average(i => rows[i].Y);
    foreach (var i in validIdx)
        rows[i].Te = map.TryGetValue(rows[i].ModelCode, out var v) ? v : prior;
}
```


In [ ]:
# GOAL: 素朴な target encoding を作り、CV が「不自然に良くなる」ところまでを実測する

prior = float(y.mean())          # 全体平均(log 空間)
print("prior(train 全体の平均 log 価格) =", round(prior, 6))

# STEP 1: train 全体で model_code ごとの y の平均を作る ← ここが後で問題になる
te_naive = pd.Series(y).groupby(train["model_code"].to_numpy()).mean()
print("\n対応表:", type(te_naive).__name__, " 水準数:", len(te_naive))
print(te_naive.head(3).round(6).to_string())

# STEP 2: 各行に貼る。列は1本しか増えない
X_te_naive = X_cat.copy()
X_te_naive["mc_te"] = train["model_code"].map(te_naive).to_numpy()
print("\nX_cat:", X_cat.shape, "→ X_te_naive:", X_te_naive.shape, "(+1 列)")

# STEP 3: 作った列と目的変数の関係を見る。ここで気づけるかどうかが分かれ目
print("corr(mc_te, y) =", round(float(np.corrcoef(X_te_naive["mc_te"], y)[0, 1]), 6))

vc = train["model_code"].value_counts()
one_key = vc[vc == 1].index[0]                       # train に1行しかない型番
row = int(np.flatnonzero(train["model_code"].to_numpy() == one_key)[0])
print(f"\ntrain に1行しかない型番 {one_key} の行(行番号 {row}):")
print(f"    その行の y           = {y[row]:.6f}")
print(f"    その行の mc_te       = {X_te_naive['mc_te'].iloc[row]:.6f}   ← 完全に一致している")
print(f"    こういう行が {int((vc == 1).sum())} 行ある(概念1 で数えた n_singleton)")

# STEP 4: CV を回す。動かしたのは「列を1本足した」だけ
score_base = cv_rmse(lgbm(), X_cat)
score_te_naive = cv_rmse(lgbm(), X_te_naive)
print(f"\nDay3 の到達点(X_cat)           CV RMSE = {score_base:.6f}")
print(f"素朴な target encoding を足す   CV RMSE = {score_te_naive:.6f}"
      f"   ← {score_base / score_te_naive:.1f}倍『良く』なった")
print("\n1列足しただけでスコアが 4 倍良くなった。unit02 の規律を思い出そう —— 喜ぶ前に疑う。")


## ④ 予測: 平均を「学習側だけ」から作ると、何が起きる?

次のセルでは、まったく同じ target encoding を **OOF 版**に直す。
`FOLDS`(CV と同じ分割)で回し、**学習側だけ**で平均を作って検証側に貼る。それ以外は何も変えない。

実行する前に予測しよう。

1. OOF 版の CV スコアは、素朴版(`0.173041`)からどこへ動く? Day3 の到達点(`0.707434`)より良い? 悪い?
2. **スムージング `m = 20` を掛けた素朴版**はどうなる? リークは直る? 部分的に直る? 直らない?
3. `model_code` の target encoding **だけ**を特徴量にした(カテゴリ列も何も無い)モデルを、
   素朴版と OOF 版で回したら? OOF 版は**定数予測(1.111857)より良くなる**と思う?
4. `category`(5水準)のような**低カーディナリティ**の列で同じことをしたら、
   素朴版と OOF 版の差はどのくらい開く?

> ヒント: CV は `GroupKFold(product_key)` で切っている。そして `model_code` は
> unit02 で確認した通り**商品をほぼ一意に指す**列だ。
> 検証 fold の行の `model_code` は、学習側の対応表に**載っているだろうか**?


In [ ]:
# GOAL: 「学習側だけで平均を作る」に変えるだけでスコアが現実に戻ることを実測する


def oof_target_encode(col, m=0.0, folds=FOLDS):
    """CV と同じ分割で、学習側だけから作った目的変数平均を検証側に貼る(OOF target encoding)。
    戻り値は (n_train,) の ndarray。"""
    keys = train[col].to_numpy()
    out = np.full(len(y), np.nan)
    for idx_tr, idx_va in folds:
        prior_tr = float(y[idx_tr].mean())                       # 学習側だけの全体平均
        yy = pd.Series(y[idx_tr])
        s = yy.groupby(keys[idx_tr]).sum()                       # 学習側だけの水準ごとの合計
        n = yy.groupby(keys[idx_tr]).size()                      # 学習側だけの水準ごとの件数
        mapping = (s + prior_tr * m) / (n + m)                   # スムージング付きの平均
        # 学習側に無かった水準は prior で埋める(map は未知キーを NaN にするため)
        out[idx_va] = pd.Series(keys[idx_va]).map(mapping).fillna(prior_tr).to_numpy()
    return out


# STEP 1: OOF 版を作る。素朴版との違いは「どのデータから平均を作ったか」だけ
te_oof = oof_target_encode("model_code")
print("te_oof:", te_oof.shape, " 未記入(NaN):", int(np.isnan(te_oof).sum()))
print("値の種類:", len(np.unique(te_oof.round(9))), "  ← 種類がごく少ない場合、その意味を考えること")
print("corr(素朴 te, y) =", round(float(np.corrcoef(train['model_code'].map(te_naive), y)[0, 1]), 6))
print("corr(OOF  te, y) =", round(float(np.corrcoef(te_oof, y)[0, 1]), 6))

X_te_oof = X_cat.copy()
X_te_oof["mc_te"] = te_oof

# STEP 2: 素朴 / スムージング付き素朴 / OOF を並べる
X_te_sm = X_cat.copy()
sm20 = (pd.Series(y).groupby(train["model_code"].to_numpy()).sum() + prior * 20) / \
       (pd.Series(y).groupby(train["model_code"].to_numpy()).size() + 20)
X_te_sm["mc_te"] = train["model_code"].map(sm20).to_numpy()

print(f"\n{'model_code の target encoding':<40}{'CV RMSE':>10}")
print("-" * 50)
print(f"{'(なし) Day3 の到達点':<36}{score_base:>10.6f}")
print(f"{'素朴(train 全体で平均)':<36}{score_te_naive:>10.6f}   ← 嘘")
print(f"{'素朴 + スムージング m=20':<36}{cv_rmse(lgbm(), X_te_sm):>10.6f}   ← まだ嘘")
print(f"{'OOF(学習側だけで平均)':<36}{cv_rmse(lgbm(), X_te_oof):>10.6f}   ← 現実")

# STEP 3: target encoding「だけ」で予測させる。特徴量はこの1列 + 数値4列
Xa = train[FEAT_NUM].copy()
Xa["mc_te"] = train["model_code"].map(te_naive).to_numpy()
Xb = train[FEAT_NUM].copy()
Xb["mc_te"] = te_oof
print(f"\n{'target encoding だけを頼りにしたモデル':<40}{'CV RMSE':>10}")
print("-" * 50)
print(f"{'素朴':<36}{cv_rmse(lgbm(), Xa):>10.6f}")
print(f"{'OOF':<36}{cv_rmse(lgbm(), Xb):>10.6f}   ← 定数予測(1.111857)より悪い")

# STEP 4: 低カーディナリティ列(category, 5水準)で同じ比較
for col in ["category", "condition"]:
    Xn = X_cat.copy()
    Xn[col + "_te"] = train[col].map(pd.Series(y).groupby(train[col].to_numpy()).mean()).to_numpy()
    Xo = X_cat.copy()
    Xo[col + "_te"] = oof_target_encode(col)
    print(f"\n{col}({int(train[col].nunique())}水準): 素朴 {cv_rmse(lgbm(), Xn):.6f} / "
          f"OOF {cv_rmse(lgbm(), Xo):.6f}   ← 差はほとんど無い")


## ⑥ 書いてみる: スムージング付きの対応表を作る関数

⑤ の `oof_target_encode` の中で、対応表を作る部分だけを**関数として切り出す**。
これは演習 `ex02` でそのまま部品になるし、実務でも「対応表を作る」と「貼る」を分けておくと、
学習時と推論時で同じ対応表を使い回せる(unit10 の artifacts の話に繋がる)。

次のセルで4つ作ろう。

**1. `smooth_target_map(keys, target, m)`** — スムージング付きの対応表を返す関数

  - 引数: `keys`(水準の Series または ndarray)、`target`(1次元 ndarray)、`m`(float)
  - 返り値: **index が水準、値がスムージング済み平均**の `pandas.Series`
  - 式: `(その水準の target の合計 + 全体平均 × m) / (その水準の件数 + m)`
  - 全体平均は `target` 全体の平均(この関数に渡されたものだけで完結させる)

**2. `te_map0`** — `smooth_target_map(train["model_code"], y, 0.0)` の結果

**3. `te_map20`** — 同じものを `m = 20.0` で作ったもの

**4. `score_naive_smooth`** — `te_map20` を**素朴に**(train 全体の対応表として)貼ったときの CV RMSE。
   下のセルに用意してある `cv_with_te_map(mapping)` に渡すだけでよい

使う道具:

- `pd.Series(np.asarray(target, dtype=float))` — ndarray を Series にする
- `.groupby(np.asarray(keys)).sum()` / `.size()` — 水準ごとの合計と件数(②の API 表)
- 2つの Series 同士の `+` `/` は **index を揃えた要素ごとの演算**(NumPy のブロードキャストと同じ発想)

3〜5行で書ける。`m=0.0` のとき `0 * prior = 0` かつ `n + 0 = n` なので、
**自然に素朴な平均に戻る**ように書けているかを確かめよう。


In [ ]:
def cv_with_te_map(mapping, col="model_code"):
    """与えられた対応表で col を置き換えた列を X_cat に足し、CV RMSE を返す(このセルは書き換えなくてよい)。"""
    try:
        Xm = X_cat.copy()
        Xm[col + "_te"] = train[col].map(mapping).astype(float).to_numpy()
        return cv_rmse(lgbm(), Xm)
    except Exception as e:
        print(f"     (cv_with_te_map の中で例外 → {type(e).__name__}: {e})")
        return None


def smooth_target_map(keys, target, m):
    """スムージング付きの目的変数平均の対応表(index=水準)を返す。"""
    # ここに書く(ヒント: pd.Series(target) を groupby(keys) して sum() と size() を取る)
    return None


te_map0 = None             # ここに書く(ヒント: smooth_target_map(train["model_code"], y, 0.0))
te_map20 = None            # ここに書く(ヒント: 同じものを m=20.0 で)
score_naive_smooth = None  # ここに書く(ヒント: cv_with_te_map(te_map20))

print("te_map0            :", None if te_map0 is None else f"{type(te_map0).__name__} 水準数={len(te_map0)}")
print("te_map20           :", None if te_map20 is None else f"{type(te_map20).__name__} 水準数={len(te_map20)}")
print("score_naive_smooth :", score_naive_smooth)
if te_map0 is not None and te_map20 is not None:
    cmp = pd.DataFrame({"m=0": te_map0, "m=20": te_map20}).head(5).round(6)
    print(cmp.to_string())


In [ ]:
# ===== チェックポイント B: target encoding =====
check("B-1 対応表の水準数", None if te_map0 is None else len(te_map0), 756,
      hint="model_code の水準数ぶんの行がある Series を返す。1661 なら行ごとの値を返している"
           "(対応表ではなく貼った結果になっている)。")

# train に1行しかない型番。m=0 の target encoding はその行自身の y と一致するはず
KEY1 = "MC-9037-0076"
check(f"B-2 m=0 のときの {KEY1} の値", series_at(te_map0, KEY1), 8.849371,
      hint="この型番は train に1行しかない。だから平均 = その行の y そのもの(8.849371)になる。"
           "None なら Series が返っていないか、index が水準になっていない。")

check(f"B-3 m=20 のときの {KEY1} の値", series_at(te_map20, KEY1), 9.019778,
      hint="prior=9.028298 に強く引き寄せられる。式は (合計 + prior*m) / (件数 + m) = "
           "(8.849371 + 9.028298*20) / (1 + 20)。8.849371 のままなら m を使っていない。")

check("B-4 m=20 の素朴 target encoding での CV RMSE", score_naive_smooth, 0.332671,
      hint="cv_with_te_map(te_map20) の戻り値。0.173041 なら te_map0(m=0)を渡している。"
           "0.707434 なら列が足されていない。")

print("\n(4つとも [OK] になったら次の概念へ)")
print("※ B-2 が今日の核心。件数1件の target encoding は『その行の答えを書き写しただけ』であり、")
print("  スムージング(B-3)はそれを薄めるだけでリーク自体は直さない。直すのは OOF 化だけ。")


---
# 概念3 — 集約特徴と日付特徴

## ① なぜ: 「1行だけ見ても分からないこと」を列にする

`views = 300` は多いのか少ないのか。それは**どのカテゴリの商品か**による。
家電の平均閲覧数が 118、本・音楽が 102 なら、同じ 300 でも意味が違う。
この「**群の中での相対的な位置**」は、1行を眺めているだけでは絶対に出てこない情報だ。

実務での典型例はいくらでもある。「この店舗の今日の売上は、その店舗の平常値の何倍か」
「このユーザーの注文金額は、そのユーザーの平均から何σ離れているか(異常検知)」
「この商品の価格は、同カテゴリの中央値の何%か(価格競争力)」。
どれも **groupby → 集計 → 元の行に貼り戻す** という同じ形をしている。

日付も同じで、`2026-01-15` という値そのものはモデルにとって意味を持たない。
そこから**年・月・曜日・起点からの経過日数**を切り出して初めて特徴量になる。
特にこのコンペでは、unit02 で実測した通り**価格が時間とともに下落している(月あたり約6%)**。
この傾きを拾うには「経過日数」という**連続値**の列が要る。

pandas の `groupby` / `agg` / `transform` / `merge` / `dt` は、今日ここで初めて本格的に使う。
C# の LINQ を書いたことがあるなら、対応は1対1で付く。


## ② 解説: `agg` は行数が減り、`transform` は行数が保たれる

### C# との対応表

| C# (LINQ) | pandas | 結果の行数 |
|---|---|---|
| `items.GroupBy(x => x.Category)` | `df.groupby("category")` | (まだ計算しない。遅延評価も同じ) |
| `.Select(g => new { Cat = g.Key, N = g.Count(), Avg = g.Average(x => x.Views) })` | `.agg(n=("record_id","size"), views_mean=("views","mean"))` | **グループ数**(5行) |
| グループ統計を**元の各要素に配り直す**(`join` し直す) | `.transform("mean")` | **元の行数**(1661行) |
| `items.Join(stats, x => x.Cat, s => s.Cat, (x, s) => ...)` | `df.merge(stats, on="category", how="left")` | 元の行数(**右表のキーが一意なら**) |
| `items.GroupBy(k).Select(g => g.OrderBy(...))` の順位付け | `.rank()` を `groupby` に掛ける | 元の行数 |

**`agg` と `transform` の違いは「戻ってくる行数」だけ**、と覚えるのが最短だ。
特徴量として列に足したいときは `transform`(行数が変わらないのでそのまま代入できる)。
集計表そのものを見たい/保存したいときは `agg`。

```
train (1661 行)                agg → (5, k)                    transform → (1661,)
┌──────────────┐        ┌──────────────────┐        ┌──────────────┐
│ 家電   views=300 │        │ 家電    mean=118.2 │        │ 118.2        │
│ ホビー views=90  │  ───►  │ ホビー  mean=107.0 │  ───►  │ 107.0        │
│ 家電   views=150 │        │ ...              │        │ 118.2        │
│ ...          │        └──────────────────┘        │ ...          │
└──────────────┘         行が「潰れる」                 行が「潰れない」
```

### API 一覧

| 用途 | API | 戻り値 | 注意 |
|---|---|---|---|
| グループ化 | `df.groupby("col")` / `df.groupby(["a","b"])` | GroupBy オブジェクト | この時点では計算されない。既定で `NaN` のキーは**除外される** |
| 名前付き集計 | `g.agg(新列名=("元列","統計名"))` | DataFrame(index = グループキー) | 統計名は `"mean" / "median" / "sum" / "size" / "count" / "min" / "max" / "std" / "nunique"`。`"size"` は行数、`"count"` は欠損を除いた個数 |
| index を列に戻す | `.reset_index()` | DataFrame(キーが普通の列になる) | `merge` の前にはほぼ必ず必要 |
| 群統計を元の行に配る | `df.groupby("a")["b"].transform("mean")` | **元の行数の Series**(index も元のまま) | そのまま `df["新列"] = ...` で代入できる |
| 結合 | `df.merge(right, on="key", how="left")` | DataFrame | **右表のキーが重複していると行が増える**。前後で `shape` を必ず print |
| 日時の部品 | `s.dt.year` / `.month` / `.day` / `.dayofweek` / `.hour` | int の Series | `dt` は**日時型の Series 専用のアクセサ**。`parse_dates` や `pd.to_datetime` を通していないと使えない |
| 日数差 | `(s - 起点).dt.days` | int の Series | 引き算の結果は timedelta 型。`.dt.days` で整数の日数になる |

> **`dt` アクセサとは** — 日時型の Series にだけ生えている「日時としての操作をまとめた名前空間」。
> C# の `DateTime` の `.Year` / `.Month` / `.DayOfWeek` プロパティを、列全体に一括適用するものだと思えばよい。
> 文字列列には `.str` という同じ形のアクセサがある(unit05 で使う)。

### 群内相対値 — いちばん効く型

集約特徴の中で最も効きやすいのは、**生の群統計そのものではなく、群統計との比・差**だ。

| 形 | 式 | 意味 |
|---|---|---|
| 比 | `x / groupby(g)[x].transform("mean")` | 群平均の何倍か |
| 差 | `x - groupby(g)[x].transform("mean")` | 群平均から何だけ離れているか |
| 標準化 | `(x - 群平均) / 群標準偏差` | 群内での偏差値 |
| 順位 | `groupby(g)[x].rank(pct=True)` | 群内での上位何%か |

> **注意: 集約する列に目的変数を使った瞬間、それは target encoding になる**(= 概念2 の話に戻る)。
> `groupby("category")["price"].transform("mean")` は便利そうに見えるが、**自分自身を含む集計**であり、
> fold の外で作れば確実にリークする。**集約に使ってよいのは説明変数だけ**、と覚えておくのが安全だ。

### 日付特徴の作り方

| 特徴 | 作り方 | このコンペで効くか |
|---|---|---|
| 年 / 月 | `.dt.year` / `.dt.month` | 弱い。しかも **test は train に無い月**なので注意(答え合わせで実測する) |
| 曜日 | `.dt.dayofweek`(月曜 = 0) | このデータには曜日効果を入れていないので効かない |
| **起点からの経過日数** | `(s - ORIGIN).dt.days` | **効く**。価格が月あたり約6%下落しているため(unit02 で実測) |
| 周期性の sin/cos | `sin(2π·月/12)`, `cos(2π·月/12)` | 月を `1..12` の整数で入れると **12月と1月が最も遠い**ことになる。円周上に置き直す変換 |

> このデータの `days` と `y` の相関は **-0.036** しかない。傾きが小さいのではなく、
> 商品ごとのばらつき(σ ≈ 0.62)がトレンド(5か月で -0.3)を覆い隠しているからだ。
> **相関が小さい = 効かない、ではない。** 他の列で説明できる部分を除いた後に効いてくる。


In [ ]:
# GOAL: agg / transform / merge の「行数がどうなるか」を shape で確かめる

# STEP 1: agg — グループごとに1行に潰す(C# の GroupBy().Select(g => new {...}))
agg_view = train.groupby("category").agg(
    n=("record_id", "size"),            # 行数
    views_mean=("views", "mean"),       # 平均
    views_max=("views", "max"),         # 最大
).reset_index()                          # index になったキーを普通の列に戻す
print("train:", train.shape, "→ agg:", agg_view.shape, " ← 行が『潰れた』(5 グループ)")
print(agg_view.round(2).to_string(index=False))
print("n の合計:", int(agg_view["n"].sum()), "= 元の行数", len(train))

# STEP 2: transform — 同じ平均を、元の行数のまま配り直す
views_mean_by_cat = train.groupby("category")["views"].transform("mean")
print("\ntransform の戻り値:", views_mean_by_cat.shape, " ← 行数が保たれている(そのまま列に代入できる)")
print(pd.DataFrame({"category": train["category"], "views": train["views"],
                    "群平均": views_mean_by_cat.round(2)}).head(5).to_string(index=False))

# STEP 3: merge — agg した表を元のテーブルに貼り戻す。行数が増えていないかを必ず確認する
merged = train.merge(agg_view, on="category", how="left")
print("\nmerge 前:", train.shape, "→ merge 後:", merged.shape,
      " 行数は変わったか:", merged.shape[0] != train.shape[0])

# STEP 4: merge の事故 — 右表のキーが重複していると行が増える
bad_right = pd.concat([agg_view, agg_view], ignore_index=True)   # わざとキーを2重にした表
bad = train.merge(bad_right, on="category", how="left")
print("右表のキーが重複していると:", train.shape, "→", bad.shape, f" ({bad.shape[0] / len(train):.0f}倍に増殖)")
print("→ merge の後は必ず shape を print する。黙って行が倍になるのが pandas でいちばん怖い事故")

# STEP 5: dt アクセサ — 日時型の Series から部品を取り出す
print("\ncollected_at の dtype:", train["collected_at"].dtype)
parts = pd.DataFrame({
    "collected_at": train["collected_at"].head(4),
    "year": train["collected_at"].dt.year.head(4),
    "month": train["collected_at"].dt.month.head(4),
    "dayofweek": train["collected_at"].dt.dayofweek.head(4),   # 月曜=0
    "days(起点から)": (train["collected_at"] - ORIGIN).dt.days.head(4),
})
print(parts.to_string(index=False))
print("\ntrain に出てくる月:", sorted(train["collected_at"].dt.month.unique().tolist()),
      " / 年:", sorted(train["collected_at"].dt.year.unique().tolist()))
print("corr(days, y) =", round(float(np.corrcoef(train["days"], y)[0, 1]), 6))


## ④ 予測: 日付を分解すると、スコアはどう動く?

次のセルでは `X_cat`(7列)に **`year` / `month` / `dow`** の3列を足して `X_date`(10列)を作り、CV を測る。
あわせて、**`days`(経過日数)を抜いたらどうなるか**を LightGBM と Ridge の両方で測る。

実行する前に予測しよう。

1. `year` / `month` / `dow` を足すと、LightGBM の CV(いま `0.707434`)はどう動く?
2. `days` を**抜く**と、LightGBM のスコアはどれくらい悪化する? Ridge はどうか?
   (①で書いた通り、価格は月あたり約6%下落している)
3. なぜ木モデルは「経過日数」のような**なめらかな直線のトレンド**を苦手とするのだろう?
   (unit03 の「決定木は軸に平行な分割しかできない」「外挿できない」を思い出そう)
4. `dow`(曜日)は効くと思う? このデータの作られ方(`make_data.py` の説明)を思い出そう。

> ヒント: `month` は `10, 11, 12, 1, 2` の5値しかない。木にとってこれは「5水準のカテゴリ」とほぼ同じで、
> `days` を粗くしたものにすぎない。**同じ情報を粒度を変えて2回入れる**と何が起きるだろうか。


In [ ]:
# GOAL: 日付特徴を足す・経過日数を抜く、をモデル2種で測って「効き方はモデルによって違う」を見る

# STEP 1: dt アクセサで3列足す
X_date = X_cat.copy()
X_date["year"] = train["collected_at"].dt.year
X_date["month"] = train["collected_at"].dt.month
X_date["dow"] = train["collected_at"].dt.dayofweek
print("X_cat:", X_cat.shape, "→ X_date:", X_date.shape)

score_date = cv_rmse(lgbm(), X_date)
print(f"\nX_cat  (7 列)  CV RMSE = {score_base:.6f}")
print(f"X_date (10 列) CV RMSE = {score_date:.6f}   ← 日付を分解して足した")

# STEP 2: days(経過日数)を抜いてみる。LightGBM と Ridge の両方で
X_nodays = X_cat.drop(columns=["days"])
X_oh_nodays = X_oh.drop(columns=["days"])
print(f"\n{'':<22}{'days あり':>12}{'days なし':>12}")
print("-" * 46)
print(f"{'LightGBM':<18}{score_base:>12.6f}{cv_rmse(lgbm(), X_nodays):>12.6f}")
print(f"{'Ridge(one-hot)':<18}{cv_rmse(ridge, X_oh):>12.6f}{cv_rmse(ridge, X_oh_nodays):>12.6f}")
print("→ LightGBM はほぼ動かない(0.7074 → 0.7080)。Ridge は 0.6379 → 0.6431 とはっきり悪化する")
print("  木は『なめらかな直線』を階段で近似するしかないので、時間トレンドを取り込むのが苦手。")
print("  線形モデルは傾きを1つの係数で表せるので、この列がそのまま効く")

# STEP 3: 月ごとの平均 log 価格を見る。トレンドは「見えるが埋もれている」
by_month = (train.assign(ym=train["collected_at"].dt.to_period("M").astype(str))
                 .groupby("ym").agg(n=("price", "size"), mean_logprice=("price", lambda s: float(np.log1p(s).mean()))))
print("\n", by_month.round(4).to_string())
print("→ 単調に下がってはいない。月ごとに商品構成が違うので、トレンドが商品ばらつきに埋もれている。")
print("  『相関が小さいから効かない』と切り捨てないこと")


## ⑥ 書いてみる: 集計表を作り、群内相対値を特徴量にする

②の表の **群内相対値**(いちばん効きやすい型)を実際に作る。
`views` を「その行のカテゴリの平均 views の何倍か」に変換する。

次のセルで4つ作ろう。

| 変数 | 中身 |
|---|---|
| `agg_cat` | `train` を `category` でグループ化した集計表。列は **`["category", "n", "views_mean", "days_max"]` の順**。`n` は行数(`"size"`)、`views_mean` は `views` の平均、`days_max` は `days` の最大。`reset_index()` してキーを列に戻すこと |
| `X_g` | `X_date` のコピーに、列 `views_rel_cat`(= 各行の `views` ÷ その行のカテゴリの平均 `views`)を足したもの(11 列) |
| `score_g` | `X_g` の CV RMSE |
| `rel_mean` | `X_g["views_rel_cat"]` の平均(`float`) |

使う道具:

- `train.groupby("category").agg(新列名=("元列", "統計名"))` — 名前付き集計(②の API 表)
- `.reset_index()` — グループキーを普通の列に戻す
- `train.groupby("category")["views"].transform("mean")` — **元の行数のまま**群平均を配る
- `X_date.copy()` — 元を壊さない

3〜5行で書ける。`rel_mean` は計算する前に**値を予想してから**実行してみよう
(「各群の中で平均で割った比」を全体で平均すると、何になるだろうか?)。


In [ ]:
agg_cat = None   # ここに書く(ヒント: train.groupby("category").agg(n=..., views_mean=..., days_max=...).reset_index())
X_g = None       # ここに書く(ヒント: X_date.copy() に transform("mean") で割った列を足す)
score_g = None   # ここに書く(ヒント: cv_rmse(lgbm(), X_g))
rel_mean = None  # ここに書く(ヒント: float(X_g["views_rel_cat"].mean()))

print("agg_cat :", None if agg_cat is None else agg_cat.shape)
if isinstance(agg_cat, pd.DataFrame):
    print(agg_cat.round(4).to_string(index=False))
print("X_g     :", None if X_g is None else X_g.shape)
print("score_g :", score_g, " rel_mean:", rel_mean)


In [ ]:
# ===== チェックポイント C: 集約特徴と日付特徴 =====
check_frame("C-1 agg_cat の形と列名", agg_cat, shape=(5, 4),
            columns=["category", "n", "views_mean", "days_max"],
            hint="groupby('category').agg(...) は 5 行。列が3列なら reset_index() を忘れている"
                 "(キーが index に入ったまま)。列名と順序は agg の引数の順で決まる。")

check("C-2 n の合計", frame_stat(agg_cat, "n", "sum"), 1661.0,
      hint="'size' は行数。全グループを足すと元の行数 1661 に戻る。'count' や 'nunique' を使うとズレる。")

check_frame("C-3 X_g の形", X_g, shape=(1661, 11),
            hint="X_date(10 列)に views_rel_cat を1本足して 11 列。行数は絶対に変わらない"
                 "(transform を使えば変わらない。merge を使うと変わり得るので shape を確認)。")

check("C-4 X_g の CV RMSE", score_g, 0.703111,
      hint="cv_rmse(lgbm(), X_g)。0.702299 なら X_date のまま測っている(列を足せていない)。")

check("C-5 群内相対値の全体平均", rel_mean, 1.0,
      hint="各群の中で『その群の平均』で割っているので、群ごとの合計が件数に一致し、"
           "全体平均はちょうど 1.0 になる。1.0 にならないなら群平均ではなく全体平均で割っている。")

print("\n(5つとも [OK] になったら次の概念へ)")
print("※ C-5 は『作った特徴量が意図通りか』を数学的に検算する習慣そのもの。")
print("  shape と、こういう不変量を必ず1つ確認してから CV を回すと、無駄な実験を減らせる。")


---
# 概念4 — Pipeline / ColumnTransformer でリークを構造的に防ぐ

## ① なぜ: 「気をつける」で防げるものは、いつか必ず漏れる

ここまでで作った前処理を数えてみよう。**標準化**(平均と標準偏差を覚える)、**欠損補完**(中央値を覚える)、
**one-hot**(水準の一覧を覚える)、**target encoding**(水準ごとの目的変数平均を覚える)。
どれも「**データを見て何かを覚え、それを適用する**」という同じ形をしている。

そして**覚える工程を fold の外でやると、検証 fold の情報が学習側に混ざる**。unit02 のリーク4類型のうちの
「前処理リーク」がこれで、概念2 の target encoding はその最も派手な例だった。

手で気をつけることもできる。だが実務のコードは、前処理が10工程になり、モデルが3種類になり、
CV が二重になり(ハイパーパラメータ探索の中に CV が入る)、半年後に別の人が触る。
**「気をつける」で守られている不変条件は、いつか必ず破られる。**

`Pipeline` と `ColumnTransformer` は、これを**構造で**解決する。
前処理を「推定器の一部」にしてしまえば、`fit` が呼ばれるのは学習側だけであることが**型として保証される**。
気をつける必要が無くなる。今日の最後は、この道具を手に入れて終わる。


## ② 解説: 前処理を「モデルの一部」にする

### C# との対応

| C# での構図 | scikit-learn |
|---|---|
| DI コンテナに `IPreprocessor` と `IModel` を**1つのコンポーネント**として登録し、外からは `IEstimator` 1本に見せる | `Pipeline([("prep", ...), ("model", ...)])` |
| ミドルウェアの合成(`app.Use(A).Use(B).Run(Handler)`) | `Pipeline` の各 step が順に `fit_transform` され、最後だけ `fit` |
| デコレータパターン(同じインターフェースで包む) | Pipeline 自身も `fit` / `predict` を持つ **estimator** |
| 差し替え可能な実装をコンストラクタで注入 | `Pipeline` の step を丸ごと入れ替えて再実験できる |
| プロパティ経由の設定(`options.Ridge.Alpha = 10`) | `pipe.set_params(model__alpha=10)`(**段名 + `__` + 引数名**) |

### `Pipeline` — 縦に繋ぐ

```python
pipe = Pipeline([("prep", 前処理), ("model", Ridge(alpha=1.0))])
pipe.fit(X_tr, y_tr)   # → prep.fit_transform(X_tr, y_tr) → model.fit(変換後, y_tr)
pipe.predict(X_va)     # → prep.transform(X_va)          → model.predict(変換後)
```

ポイントは、**学習時は `fit_transform`、予測時は `transform`** が呼ばれることだ。
つまり「覚える」のは `fit` のときだけ。`cross_val_score(pipe, X, y, cv=...)` に渡せば、
各 fold で `pipe.fit(学習fold)` が呼ばれるので、**前処理が学習 fold だけで fit されることが自動的に保証される**。

### `ColumnTransformer` — 横に分ける

列によって当てたい前処理が違う。数値列には「欠損補完 → 標準化」、カテゴリ列には「one-hot」。
これをやるのが `ColumnTransformer` で、**列の部分集合ごとに別の変換器を当てて、結果を横に連結する**。

```python
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), FEAT_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), FEAT_CAT),
])
#         ^名前          ^変換器(Pipeline でもよい)                            ^当てる列のリスト
```

| 用途 | API | 注意 |
|---|---|---|
| 列ごとに変換器を割り当て | `ColumnTransformer([(名前, 変換器, 列リスト), ...])` | 列リストに**入れなかった列は捨てられる**(既定 `remainder="drop"`)。残したいなら `remainder="passthrough"` |
| 変換後の列名 | `prep.get_feature_names_out()` | `fit` の後にだけ使える |
| 段の設定を触る | `pipe.set_params(model__alpha=10)` / `prep__cat__min_frequency=5` | `__` でネストを辿る |
| DataFrame で受け取る | `prep.set_output(transform="pandas")` | 既定は ndarray / 疎行列 |

### 「正しい CV」と「間違った CV」

| 書き方 | 何が起きるか |
|---|---|
| `cross_val_score(pipeline, X_raw, y, cv=..., groups=...)` | **正しい**。fold ごとに前処理から学習し直す |
| `X_pre = prep.fit_transform(X_raw, y)` してから `cross_val_score(model, X_pre, ...)` | **間違い**。`X_pre` は全データを見た前処理を通っている = 検証 fold の情報が学習側に混ざっている |

この2つ、**書き間違えても例外は出ない**。出るのは「ちょっと良いスコア」だけだ。
だからこそ構造で防ぐ。次のセルで、両方の書き方を実際に走らせて差を測る。

> **`cross_val_score` のスコアの定義に注意** — これは **fold ごとのスコアの平均**を返す。
> 我々の `cv_rmse` は **OOF 予測をまとめてから1回 RMSE を取る**。どちらも妥当だが数字はわずかにズレる
> (今日の例では 0.637587 と 0.637841)。**比較するときは必ず同じ定義で揃えること。**


In [ ]:
# GOAL: ColumnTransformer で「列ごとに違う前処理」を組み、Pipeline ごと CV に渡す形を作る

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.model_selection import cross_val_score

# STEP 1: 生のまま(文字列のまま)の入力。ここから先の変換は全部 Pipeline の中でやる
X_raw = train[FEAT_NUM + FEAT_CAT].copy()
print("X_raw:", X_raw.shape, " dtype:", {c: str(X_raw[c].dtype) for c in FEAT_CAT})

# STEP 2: 列ごとに別の前処理を当てる
num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),   # 中央値で欠損を埋める
                     ("sc", StandardScaler())])                    # 平均0・分散1にする
prep_oh = ColumnTransformer([
    ("num", num_pipe, FEAT_NUM),                                   # 数値4列 → 補完 → 標準化
    ("cat", OneHotEncoder(handle_unknown="ignore"), FEAT_CAT),     # カテゴリ3列 → one-hot
])
pipe_oh = Pipeline([("prep", prep_oh), ("model", Ridge(alpha=1.0))])
print("\npipe_oh の段:", [name for name, _ in pipe_oh.steps])
print("prep の枝    :", [name for name, _, cols in prep_oh.transformers])

# STEP 3: 出力が何列になるか(確認のためだけに全データで fit している。学習には使わない)
X_out = clone(prep_oh).fit_transform(X_raw, y)
print("\nColumnTransformer の出力:", X_out.shape, type(X_out).__name__,
      "= 数値4 + one-hot", sum(int(train[c].nunique()) for c in FEAT_CAT))
print("列名:", list(clone(prep_oh).fit(X_raw, y).get_feature_names_out()))

# STEP 4: Pipeline ごと CV に渡す。これが「正しい」書き方
score_pipe_oh = -cross_val_score(pipe_oh, X_raw, y, cv=gkf, groups=groups,
                                 scoring="neg_root_mean_squared_error").mean()
print(f"\ncross_val_score(pipeline ごと)     = {score_pipe_oh:.6f}  ← fold ごとの RMSE の平均")
print(f"cv_rmse(pipeline ごと・OOF まとめて) = {cv_rmse(pipe_oh, X_raw):.6f}  ← 定義が違うので少しズレる")

# STEP 5: 「間違った」書き方 — 前処理を CV の外でやる
X_pre = clone(prep_oh).fit_transform(X_raw, y)      # 全データを見て fit してしまった
score_outside = -cross_val_score(Ridge(alpha=1.0), pd.DataFrame(X_pre), y, cv=gkf, groups=groups,
                                 scoring="neg_root_mean_squared_error").mean()
print(f"\n前処理を CV の外でやった場合         = {score_outside:.6f}")
print(f"差 = {abs(score_pipe_oh - score_outside):.6f}   ← 標準化と one-hot だけなら差はほぼ出ない")
print("→ ここが落とし穴。『やってみたけど差が無かったから大丈夫』という経験則が、次のセルで裏切られる")


## ④ 予測: 目的変数を使う前処理を1つ混ぜると?

③ の結論は「標準化と one-hot だけなら、CV の中でやっても外でやっても差は出ない」だった。
理由も明快で、**標準化も one-hot も目的変数 `y` を一切見ていない**からだ。
漏れているのは「検証 fold の `views` の平均値」程度の、ごく薄い情報にすぎない。

では、`ColumnTransformer` に **`TargetEncoder`(sklearn 版の target encoding)を1枝足す**とどうなるか。
対象列は概念2 と同じ `model_code`(756水準)。

実行する前に予測しよう。

1. **Pipeline ごと** CV に渡した場合、スコア(いま `0.637587`)はどう動く? 良くなる? **悪くなる**?
2. **前処理を CV の外でやった**場合はどうなる? ③ではほぼ同じだった2つの数字は、どのくらい開く?
3. sklearn の `TargetEncoder` は `fit_transform` の中で**内部的に交差適合**(内部で分割して自分の答えを見ないように)する。
   ならば Pipeline に入れておけば安全だろうか? **このデータの分割は `GroupKFold(product_key)` だった**ことを思い出そう。
4. `TargetEncoder` を `fit(X, y)` してから `transform(X)` した結果と、`fit_transform(X, y)` の結果は
   **同じになる**と思う?

> ヒント: 3 が今日いちばん実務的な問いだ。「ライブラリが内部でよしなにやってくれる」は、
> **どういう分割で**よしなにやっているかを確認するまで信用してはいけない。


In [ ]:
# GOAL: 目的変数を使う前処理を混ぜた瞬間、「CV の中か外か」が致命的な差になることを実測する

X_raw_te = train[FEAT_NUM + FEAT_CAT + ["model_code"]].copy()
print("X_raw_te:", X_raw_te.shape)

# STEP 1: ColumnTransformer に TargetEncoder の枝を追加する
prep_te = ColumnTransformer([
    ("num", num_pipe, FEAT_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), FEAT_CAT),
    ("te", TargetEncoder(random_state=0), ["model_code"]),      # ← 目的変数を見る前処理
])
pipe_te = Pipeline([("prep", prep_te), ("model", Ridge(alpha=1.0))])

# STEP 2: 正しい書き方 — Pipeline ごと CV に渡す
score_te_pipe = -cross_val_score(pipe_te, X_raw_te, y, cv=gkf, groups=groups,
                                 scoring="neg_root_mean_squared_error").mean()

# STEP 3: 間違った書き方 — 前処理を CV の外でやる
prep_fitted = clone(prep_te)
X_pre_te = prep_fitted.fit_transform(X_raw_te, y)               # 全データで fit してしまった
score_te_outside = -cross_val_score(Ridge(alpha=1.0), pd.DataFrame(X_pre_te), y, cv=gkf, groups=groups,
                                    scoring="neg_root_mean_squared_error").mean()

print(f"\n{'':<36}{'TE なし':>12}{'TE あり':>12}")
print("-" * 62)
print(f"{'Pipeline ごと CV(正しい)':<30}{score_pipe_oh:>12.6f}{score_te_pipe:>12.6f}")
print(f"{'前処理を CV の外(間違い)':<30}{score_outside:>12.6f}{score_te_outside:>12.6f}")
print(f"{'差':<32}{abs(score_pipe_oh - score_outside):>12.6f}{abs(score_te_pipe - score_te_outside):>12.6f}")
print("\n→ TE なしでは差が 0.000000。TE ありでは 0.302589 開いた。同じコードの書き方の違いだけで")

# STEP 4: fit_transform と transform は別物(TargetEncoder の交差適合)
te = TargetEncoder(random_state=0)
a = te.fit_transform(train[["model_code"]], y).ravel()          # 内部で交差適合する
b = te.fit(train[["model_code"]], y).transform(train[["model_code"]]).ravel()   # 全データの平均をそのまま
print(f"\nfit_transform の結果と y の相関 = {float(np.corrcoef(a, y)[0, 1]):.6f}  ← 交差適合されている")
print(f"transform      の結果と y の相関 = {float(np.corrcoef(b, y)[0, 1]):.6f}  ← 自分の答えを見ている")
print("同じ変換器でも fit_transform と transform で結果が違う。これは『バグ』ではなく仕様")

# STEP 5: それでも Pipeline ごとの TE ありは、TE なしより悪い。なぜか
print(f"\nTE あり(Pipeline ごと)= {score_te_pipe:.6f} は TE なし {score_pipe_oh:.6f} より悪い。")
print("sklearn の TargetEncoder は内部で交差適合するが、その分割は KFold であって")
print("**GroupKFold ではない**。同一商品(product_key)が内部分割をまたぐので、学習 fold の中では")
print("まだ少し『自分の答え』が見えている。だからモデルはこの列を信じ、検証 fold で裏切られる。")
print("→ グループ構造のあるデータでは、概念2 で書いた **自前の OOF target encoding** が必要になる")


## ⑥ 書いてみる: 今日の特徴量を全部載せた Pipeline を組む

最後に、今日作った要素を1本の `Pipeline` にまとめる。
数値列は「欠損補完 → 標準化」、**カテゴリ列に加えて `month` と `dow` も one-hot** にして線形モデルに渡す。

> `month` と `dow` を**整数のまま**入れると、線形モデルは「月が1増えると価格が w 増える」としか表せない。
> 12月(=12)と1月(=1)が最も遠い、という②で触れた問題そのものだ。だから one-hot 側に回す。
> ……この判断が本番でどう出るかは、このあとの「答え合わせ」で分かる。

次のセルで4つ作ろう(`raw_final` と `CAT_FINAL` は用意してある)。

| 変数 | 中身 |
|---|---|
| `prep_final` | `ColumnTransformer`。枝は**この順・この名前**で2本 — `("num", num_pipe, FEAT_NUM)` と `("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FINAL)` |
| `pipe_final` | `Pipeline`。段は**この順・この名前**で2段 — `("prep", prep_final)` と `("model", Ridge(alpha=1.0))` |
| `n_out` | `prep_final` の**出力列数**(`int`)。確認のため全データで `fit_transform` してよい(学習には使わない) |
| `score_final` | `cross_val_score(pipe_final, raw_final, y, cv=gkf, groups=groups, scoring="neg_root_mean_squared_error")` の平均に **マイナスを付けた値** |

使う道具は ③ ⑤ とまったく同じ。段の名前を `"prep"` / `"model"` にするのは、
チェックセルが `pipe_final.get_params()["model__alpha"]` で設定を引くためだ
(段名 + `__` + 引数名、という C# の設定バインディングのような規約)。

4〜6行で書ける。`n_out` を計算する前に、**いくつになるかを暗算してから**実行しよう
(数値列の本数 + カテゴリ列それぞれの水準数の合計)。


In [ ]:
# 与えられた材料(ここは書き換えなくてよい)
raw_final = train[FEAT_NUM + FEAT_CAT].copy()
raw_final["month"] = train["collected_at"].dt.month
raw_final["dow"] = train["collected_at"].dt.dayofweek
CAT_FINAL = FEAT_CAT + ["month", "dow"]
print("raw_final:", raw_final.shape, " one-hot に回す列:", CAT_FINAL)
print("水準数:", {c: int(raw_final[c].nunique()) for c in CAT_FINAL})

prep_final = None   # ここに書く(ヒント: ColumnTransformer([("num", num_pipe, FEAT_NUM), ("cat", OneHotEncoder(...), CAT_FINAL)]))
pipe_final = None   # ここに書く(ヒント: Pipeline([("prep", prep_final), ("model", Ridge(alpha=1.0))]))
n_out = None        # ここに書く(ヒント: clone(prep_final).fit_transform(raw_final, y).shape[1] を int に)
score_final = None  # ここに書く(ヒント: -cross_val_score(pipe_final, raw_final, y, cv=gkf, groups=groups, scoring="neg_root_mean_squared_error").mean())

print("\nprep_final :", type(prep_final).__name__)
print("pipe_final :", type(pipe_final).__name__)
print("n_out      :", n_out)
print("score_final:", score_final)


In [ ]:
# ===== チェックポイント D: Pipeline / ColumnTransformer =====
check("D-1 ColumnTransformer の出力列数", n_out, 29,
      hint="数値4 + one-hot(category 5 + site 4 + condition 4 + month 5 + dow 7 = 25)= 29。"
           "17 なら month / dow を one-hot に入れていない。31 なら month/dow が数値側にも残っている。")

check("D-2 枝の数", None if prep_final is None else len(getattr(prep_final, "transformers", []) or []), 2,
      hint="('num', ...) と ('cat', ...) の2本。ColumnTransformer([...]) にタプルのリストを渡す。")

check("D-3 最終段の Ridge の alpha", param_of(pipe_final, "model__alpha"), 1.0,
      hint="Pipeline の段名を 'model' にする(Pipeline([('prep', ...), ('model', Ridge(alpha=1.0))]))。"
           "None なら段名が違うか、make_pipeline を使って自動命名(ridge__alpha)になっている。")

check("D-4 pipeline ごとの CV RMSE", score_final, 0.642337,
      hint="cross_val_score は『高いほど良い』向きなので neg_root_mean_squared_error が返るのは負値。"
           "マイナスを付けて平均する。-0.642337 のままなら符号が逆。0.637587 なら month/dow を入れていない。")

print("\n(4つとも [OK] になったら答え合わせへ)")
print("※ D-4 は 0.642337 で、month / dow を入れない 0.637587 より **悪い**。")
print("  『特徴量を足したのに悪くなった』は普通に起きる。次のセルで、本番ではもっと悪いことが起きる。")


---
## 答え合わせ: 今日いちばん CV が良かったモデルは、本番でも良かったか

今日は CV を何度も動かした。最後にやるべきことは1つ —— **その CV は本番を言い当てたか**の確認だ。

教材なので `test` の正解を見せる。次のセルは、今日作ったモデルを**全データで再学習**して `test` を予測し、
**CV と本番 LB を横に並べる**。

| モデル | 今日どこで作ったか |
|---|---|
| 定数(train の平均) | 下限の基準 |
| LightGBM + native categorical | Day3 の到達点(概念1) |
| LightGBM + 日付特徴 | 概念3 ⑤ |
| Pipeline(one-hot + Ridge) | 概念4 ③ |
| **LightGBM + 素朴な target encoding** | **概念2 ③ — CV が 0.173 まで下がったやつ** |
| Pipeline(month/dow の one-hot 込み) | 概念4 ⑦ |

見るポイント:

1. **CV が最良だったモデルは、LB でも最良か。** 特に `0.173041` を出したモデルがどこに着地するか
2. `test` の `model_code` のうち、**train に存在するものは何%か**
3. 概念4 ⑦ の「`month` を one-hot にする」判断が、**未来の test** でどう出るか
   (`train` の月は 10,11,12,1,2 で、`test` の月は 3 だ)


In [ ]:
# GOAL: 全データ再学習 → test を予測 → 「CV は本番を言い当てたか」を1枚の表で確認する

ANSWER = DATA.resolve().parents[1] / ".solutions" / DATA.resolve().parent.name / "_answer.csv"
test = pd.read_csv(DATA / "test.csv", parse_dates=["collected_at"])
test["days"] = (test["collected_at"] - ORIGIN).dt.days           # 起点は train と同じ
print("train 期間:", train["collected_at"].min().date(), "〜", train["collected_at"].max().date())
print("test  期間:", test["collected_at"].min().date(), "〜", test["collected_at"].max().date(), " ← 未来")
print("train の月:", sorted(train["collected_at"].dt.month.unique().tolist()),
      " / test の月:", sorted(test["collected_at"].dt.month.unique().tolist()))
known = float(test["model_code"].isin(train["model_code"]).mean())
print(f"test の model_code のうち train に存在する割合: {known:.1%}  ← target encoding が効く余地")

if not ANSWER.exists():
    print("\n答えファイルが見つかりません(このセルはスキップして構いません):", ANSWER)
else:
    y_test = np.log1p(pd.read_csv(ANSWER).set_index("record_id")
                      .loc[test["record_id"], "price"].to_numpy(dtype=float))

    def rmse(a, b):
        return float(np.sqrt(np.mean((np.asarray(a, dtype=float) - np.asarray(b, dtype=float)) ** 2)))

    # STEP 1: test 側の特徴量を train と同じ形に揃える(unit03 でやった突き合わせ)
    def make_cat(df):
        Xd = df[FEAT_NUM + FEAT_CAT].copy()
        for c in FEAT_CAT:
            Xd[c] = pd.Categorical(Xd[c], categories=X_cat[c].cat.categories)   # 水準の一覧も揃える
        return Xd

    Xte_cat = make_cat(test)
    Xte_date = Xte_cat.copy()
    Xte_date["year"] = test["collected_at"].dt.year
    Xte_date["month"] = test["collected_at"].dt.month
    Xte_date["dow"] = test["collected_at"].dt.dayofweek
    Xte_raw = test[FEAT_NUM + FEAT_CAT].copy()
    Xte_final = Xte_raw.copy()
    Xte_final["month"] = test["collected_at"].dt.month
    Xte_final["dow"] = test["collected_at"].dt.dayofweek
    # 素朴 target encoding: train 全体の対応表を test に貼る。未知の型番は prior で埋める
    Xte_te = Xte_cat.copy()
    Xte_te["mc_te"] = test["model_code"].map(te_naive).fillna(prior).to_numpy()
    print("\nXte_cat:", Xte_cat.shape, " Xte_date:", Xte_date.shape,
          " Xte_final:", Xte_final.shape, " Xte_te:", Xte_te.shape)

    rows = [
        ("定数(train の平均)", 1.111857, np.full(len(test), prior)),
        ("LightGBM + native cat", score_base, lgbm().fit(X_cat, y).predict(Xte_cat)),
        ("LightGBM + 日付特徴", score_date, lgbm().fit(X_date, y).predict(Xte_date)),
        ("Pipeline(one-hot + Ridge)", score_pipe_oh, clone(pipe_oh).fit(X_raw, y).predict(Xte_raw)),
        ("LightGBM + 素朴 TE", score_te_naive, lgbm().fit(X_te_naive, y).predict(Xte_te)),
    ]
    if pipe_final is not None and score_final is not None:
        rows.append(("Pipeline(month/dow one-hot)", score_final,
                     clone(pipe_final).fit(raw_final, y).predict(Xte_final)))

    print(f"\n{'モデル':<34}{'CV RMSE':>10}{'本番 LB':>10}")
    print("-" * 54)
    for name, cv, pred in rows:
        print(f"{name:<30}{cv:>10.6f}{rmse(y_test, pred):>10.6f}")

    print("\n※ 素朴 target encoding: CV では今日の全モデル中ぶっちぎりの1位(0.173041)。")
    print("   本番では定数予測(1.174334)とほぼ変わらない 1.091404。**CV が丸ごと嘘だった**。")
    print("   理由は上に出ている —— test の model_code の 96% は train に存在しない。")
    print("   train でだけ効く『答えを書き写した列』を、モデルは全力で信じてしまっていた。")
    print("※ month / dow の one-hot: test の月は 3 で train に存在しない。")
    print("   handle_unknown='ignore' なので test 側は month の列が**全部ゼロ**になる = 情報ゼロ。")
    print("   だから連続値の days(経過日数)の方が、未来を予測するときは強い(unit03『木は外挿できない』の線形版)。")
    print("※ それ以外の4本は CV と LB の順位が一致している。これが『信じられる CV』の状態。")


---
## 振り返り(自己評価 + TIL)

以下に**1〜2文ずつ**、自分の言葉で書いてみよう。書いた内容はセッション終了時の学習ノートと
スキルレベルの判定に使う(空欄でも先に進めるが、言語化すると定着が大きく変わる)。

**1. 今日学んだことを自分の言葉で:**

> (ここに書く)

**2. 難しかったこと・まだあやふやなこと:**

> (ここに書く)

**3. Feynman チェック — 次の5つに、資料を見ずに答えられる?**

- 「木モデルには one-hot が不利で、線形モデルには必須」なのはなぜか、両者の**決定境界の作り方**に触れて説明できる?
- `train` に1行しかないカテゴリの target encoding の値は、**何と一致する**? それはなぜ問題なのか?
- スムージングは target encoding のリークを直す? 直さないとしたら、**何を**直しているのか?
- `agg` と `transform` の違いを、**戻り値の行数**で説明できる? どちらを特徴量作りに使う?
- 同僚が「前処理を先に済ませてから `cross_val_score` を回しています」と言ってきた。
  **どんな前処理なら問題なく、どんな前処理なら致命的か**を、今日の実測(0.000000 と 0.302589)に触れて説明できる?

> (ここに書く)


---
## まとめ

### 今日学んだこと

| # | 概念 | 一言でいうと |
|---|---|---|
| 1 | カテゴリ変数の表現 | one-hot / ordinal / count / native categorical / target encoding。**それぞれが違う仮定を置いている** |
| 2 | one-hot と木 | 1列あたりの情報が薄まり、同じ分割に深い木が要る。**756水準を one-hot にしたら 17→773 列でスコアは完全に同一**だった |
| 3 | one-hot と線形 | **必須**。ordinal にすると 0.637878 → 1.075022(定数予測とほぼ同じ)まで壊れた |
| 4 | native categorical | 列を増やさず集合の分割を直接学ぶ。**高カーディナリティで効く** |
| 5 | count encoding | 「珍しさ」を仮定する。今回は効かなかった(0.711380)。**効かないと判定できたのも進捗** |
| 6 | **target encoding** | カテゴリ → その水準の目的変数の平均。**最も効き、最も事故る** |
| 7 | **素朴な TE はリークする** | 件数1件の水準では、**その行自身の y を書き写している**。CV 0.707434 → **0.173041** |
| 8 | **OOF target encoding** | 学習側だけで平均を作り、検証側に貼る。CV は 0.704105 に戻る = **これが現実の値** |
| 9 | スムージング | `(合計 + prior×m) / (件数 + m)`。少数水準のノイズ対策であって、**リーク対策ではない**(m=20 でも 0.332671) |
| 10 | `groupby().agg()` | C# の `GroupBy().Select(g => new {...})`。**行がグループ数に潰れる** |
| 11 | `transform()` | 群統計を**元の行数のまま**配る。だから特徴量作りはこちら |
| 12 | `merge()` | 右表のキーが重複していると**黙って行が増える**。前後で必ず `shape` を print |
| 13 | 群内相対値 | 群平均との**比・差・順位**が最も効く型。集約に目的変数を使うと target encoding になる(= リーク) |
| 14 | `dt` アクセサ | 日時型 Series 専用の名前空間。`year` / `month` / `dayofweek` / `(s - 起点).dt.days` |
| 15 | 経過日数 | 相関は -0.036 しかないのに Ridge では 0.643130 → 0.637878 と効いた。**相関が小さい ≠ 効かない** |
| 16 | 木と時間トレンド | 木はなめらかな直線を階段で近似するしかない。日付列は**線形モデルの方が素直に使える** |
| 17 | `Pipeline` | 前処理を推定器の一部にする。**`fit` が学習側だけで呼ばれることが構造的に保証される** |
| 18 | `ColumnTransformer` | 列の部分集合ごとに別の変換器。列リストに入れ忘れた列は**黙って捨てられる** |
| 19 | 前処理リークの大きさ | `y` を見ない前処理(標準化・one-hot)なら差 **0.000000**。`y` を見る前処理を1つ混ぜた瞬間 **0.302589** |
| 20 | ライブラリを疑う | sklearn の `TargetEncoder` は内部で交差適合するが、それは **KFold であって GroupKFold ではない** |
| 21 | 未知カテゴリ | `handle_unknown="ignore"` の未知水準は**全部ゼロの行**になる。`month` を one-hot にしたら test(3月)が情報ゼロになった |
| 22 | 今日の答え合わせ | CV 最良(0.173041)のモデルが LB では最下位級(1.091404)。**CV が丸ごと嘘だった** |

### 今日いちばん持ち帰るべき1行

**目的変数から作った特徴量は、「どのデータから作ったか」を言えないなら使ってはいけない。**

### この先どこで使うか(先読み)

- **unit05(テキスト)** — `TfidfVectorizer` も `fit` / `transform` を持つ変換器。
  **語彙を train だけで fit する**のは、今日の「前処理は学習 fold だけで fit」とまったく同じ話。
  `Pipeline` に載せれば同じように構造で守れる。
- **unit06(名寄せ)** — ペアの特徴量を作る工程が、今日の `groupby` / `merge` の応用そのものになる。
  そして unit02 から `product_key` として与えられていた group キーを、いよいよ**自分で作る**。
- **unit07 / unit09(深層)** — 前処理を「モデルの一部」にする発想は、
  トークナイザや画像 transform を Dataset に注入する設計に直結する。
  「学習用は拡張あり・検証用は拡張なし」を取り違えるのは、今日の前処理リークの深層版だ。
- **unit10(キャップストーン)** — 「対応表を作る」と「貼る」を分ける書き方(概念2 ⑦)が、
  **学習と推論の分離**(fit 済みエンコーダを artifacts として保存する)に直結する。
- **実務** — 新しい列を1本足すたびに問うこと: **(1) その列は目的変数から作られていないか
  (2) その列は予測時点で手に入るか (3) その列を作る統計は、どのデータから計算したか**。
  今日の3つ目が新しく増えた問いだ。

### 次にやること

**演習 `ex01_categorical_encoding` へ進もう。lesson.ipynb を見ながらで OK。**
思い出せない API があれば ② の表に戻ればいい。暗記ではなく、**どこを見れば分かるか**を覚えているのが実務の状態だ。

演習は4本:

| 演習 | 内容 |
|---|---|
| `ex01_categorical_encoding` | 4つのエンコーディングを実装し、train/test の列整合まで通す |
| `ex02_oof_target_encoding` | OOF target encoding をスムージング込みで自作する(今日の ⑦ の一般化) |
| `ex03_groupby_and_datetime` | 集約特徴・群内相対値・日付特徴をまとめて作る |
| `ex04_capstone` | 特徴量生成から Pipeline での提出まで一気通貫 |
